# Lab 03: Prompt Engineering Patterns

## Overview
In this lab you will master prompt engineering techniques for Amazon Bedrock including zero-shot, few-shot, chain-of-thought, system prompts, structured output extraction, prompt templates, and prompt chaining. You will use the Converse API with Claude to explore how each technique changes model behavior and output quality.

## Learning Objectives
- Write effective zero-shot prompts and understand when they are sufficient
- Improve output quality with few-shot examples for classification and generation tasks
- Apply chain-of-thought (CoT) prompting to improve accuracy on reasoning tasks
- Use system prompts via the Converse API to control model persona and behavior
- Extract structured JSON output reliably from foundation models
- Chain multiple model calls together to decompose complex tasks into steps

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers Task 2.3 (prompt engineering techniques for optimizing model output quality and reliability).

## Architecture Diagram
![Lab 03 Architecture](../assets/diagrams/lab-03-prompt-engineering.png)

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3, json, os

REGION = "us-east-1"

# SageMaker Studio sets AWS_DEFAULT_REGION and mounts /opt/ml
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")

# Claude Sonnet 4.5 — strong at both reasoning and creative tasks, good for prompt engineering demos
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

def invoke(prompt, system=None, max_tokens=512, temperature=0.7):
    """Reusable helper to call Claude via the Converse API."""
    kwargs = {
        "modelId": MODEL_ID,
        # Converse API uses a unified messages format across all providers
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature}
    }
    if system:
        # system parameter is separate from messages — processed with higher priority by the model
        kwargs["system"] = [{"text": system}]
    response = bedrock_runtime.converse(**kwargs)
    return response["output"]["message"]["content"][0]["text"]

print(f"Model: {MODEL_ID}")
print("Helper function ready.")

## A. Zero-Shot Prompting

**Zero-shot prompting** gives the model a task with no examples -- it relies entirely on pre-training knowledge.

- **Works well for:** straightforward tasks (summarization, translation, simple classification)
- **Struggles with:** specific output formats, ambiguous categories, domain-specific tasks
- **Try first** -- escalate to few-shot only when zero-shot output is inconsistent

### Zero-Shot Classification

No examples provided -- the model must rely on its pre-training knowledge of AWS services.

In [ ]:
# Zero-shot classification — no examples, model relies on pre-training
descriptions = [
    "A service that lets you run containers without managing servers or clusters.",
    "A fully managed graph database engine optimized for storing billions of relationships.",
    "A service that provides on-demand delivery of compute power, database storage, and other resources.",
    "A managed service for training and deploying machine learning models at scale.",
]

for desc in descriptions:
    prompt = f"""Classify the following AWS service description into exactly one category:
compute, database, storage, networking, ai/ml, security, or analytics.

Description: {desc}

Category:"""
    # temperature=0 for deterministic output in classification
    result = invoke(prompt, temperature=0.0)
    print(f"Description: {desc[:70]}...")
    print(f"Category:    {result.strip()}\n")

### Zero-Shot Summarization

Summarization is one of the strongest zero-shot capabilities of LLMs -- no examples needed.

In [ ]:
text = """Amazon Bedrock is a fully managed service that makes foundation models from
leading AI companies available through a single API. With Bedrock, you can choose
from a wide range of foundation models to find the one best suited for your use
case. Bedrock offers serverless access, so you do not need to manage any
infrastructure. You can customize foundation models with your own data using
techniques like fine-tuning and Retrieval Augmented Generation (RAG). Bedrock also
provides built-in guardrails for responsible AI, including content filtering,
personally identifiable information redaction, and topic-based denial policies.
The service integrates with other AWS services like S3, Lambda, and CloudWatch
for building end-to-end generative AI applications."""

prompt = f"Summarize the following text in exactly 2 sentences:\n\n{text}"
# temperature=0.3 — low randomness for factual summarization, slight variation allowed
summary = invoke(prompt, temperature=0.3)
print("Summary:")
print(summary)

## B. Few-Shot Prompting

**Few-shot prompting** provides 2-3 examples of the desired input-output behavior before the actual task. The examples teach format, tone, and classification criteria -- no fine-tuning required.

**How to choose good examples:**
- **Diverse** -- cover different categories and edge cases
- **Representative** -- match real production inputs
- **Consistent format** -- every example follows the exact same structure
- **Balanced** -- at least one example per category when possible

### Few-Shot Classification

Same task as Section A, but now with 3 examples that demonstrate the expected format. Compare the output consistency.

In [ ]:
# Few-shot classification — 3 examples teach the model the exact output format
few_shot_prompt = """Classify the AWS service description into exactly one category.
Respond with only the category name, nothing else.

Categories: compute, database, storage, networking, ai/ml, security, analytics

---
Description: A web service that provides resizable compute capacity in the cloud, allowing you to launch virtual servers.
Category: compute

---
Description: A fast, flexible NoSQL database service for single-digit millisecond performance at any scale.
Category: database

---
Description: An object storage service offering industry-leading scalability, data availability, and security.
Category: storage

---
Description: {description}
Category:"""

# Same descriptions as zero-shot — compare output consistency
descriptions = [
    "A service that lets you run containers without managing servers or clusters.",
    "A fully managed graph database engine optimized for storing billions of relationships.",
    "A service that provides on-demand delivery of compute power, database storage, and other resources.",
    "A managed service for training and deploying machine learning models at scale.",
]

for desc in descriptions:
    prompt = few_shot_prompt.format(description=desc)
    # temperature=0 for deterministic classification
    result = invoke(prompt, temperature=0.0)
    print(f"Description: {desc[:70]}...")
    print(f"Category:    {result.strip()}\n")

### Comparing Zero-Shot vs Few-Shot

The key difference is **output consistency**. Zero-shot might return "Compute", "compute", or "This is a compute service". Few-shot teaches the model the exact format you expect -- critical for production where downstream code parses the output programmatically.

## C. Chain-of-Thought (CoT) Prompting

**Chain-of-thought prompting** forces the model to show step-by-step reasoning before the final answer. This dramatically improves accuracy on multi-step tasks (math, logic, cost estimation).

**Why it works:** LLMs generate tokens sequentially. Each intermediate reasoning step becomes context for the next, reducing compounding errors.

Two approaches:
- **Zero-shot CoT** -- append "Let's think step by step" to the prompt
- **Structured CoT** -- provide a specific reasoning template to follow

### Direct Answer (No CoT)

A multi-step AWS cost estimation with a direct question -- no reasoning steps.

In [ ]:
# Direct question — model must jump to the answer without showing work
problem = """A company runs 3 EC2 instances at $0.10/hour each, 24/7 for a 30-day month.
They also use 500 GB of S3 storage at $0.023 per GB/month and make 1 million
S3 GET requests at $0.0004 per 1000 requests.
What is the total monthly cost?

Answer with just the dollar amount."""

# temperature=0 for deterministic math output
direct_answer = invoke(problem, temperature=0.0)
print("Direct answer (no CoT):")
print(direct_answer)

### With Chain-of-Thought

Same question, but with structured reasoning steps. Each step's output provides context for the next.

In [ ]:
# Structured CoT — each step builds on the previous, reducing compounding errors
problem_cot = """A company runs 3 EC2 instances at $0.10/hour each, 24/7 for a 30-day month.
They also use 500 GB of S3 storage at $0.023 per GB/month and make 1 million
S3 GET requests at $0.0004 per 1000 requests.
What is the total monthly cost?

Let's think step by step:
1. First, calculate the EC2 cost
2. Then, calculate the S3 storage cost
3. Then, calculate the S3 request cost
4. Finally, add them all together

Show your work for each step, then give the final total."""

# temperature=0 for deterministic math; max_tokens=1024 to allow full reasoning chain
cot_answer = invoke(problem_cot, temperature=0.0, max_tokens=1024)
print("Chain-of-thought answer:")
print(cot_answer)

# Expected: 3 × $0.10 × 24 × 30 = $216.00
#           500 × $0.023         = $11.50
#           1,000,000/1000 × $0.0004 = $0.40
#           Total = $227.90
print("\n--- Expected answer: $227.90 ---")

### Why CoT Works

- **Intermediate tokens provide context** -- each step informs the next, reducing errors
- **Errors become visible** -- you can pinpoint where reasoning breaks down
- **Complex tasks decompose naturally** -- multi-step problems become sequences of simple steps

> **Exam tip:** For questions about improving accuracy on reasoning or multi-step tasks, **chain-of-thought** is almost always the answer.

## D. System Prompts

The Converse API `system` parameter sets model behavior, persona, and constraints **before** any user message. It is processed separately from user input, giving it stronger influence.

Use system prompts for:
- **Persona** -- "You are an AWS Solutions Architect"
- **Output format** -- "Respond only in bullet points"
- **Guardrails** -- "Never discuss competitor products"
- **Tone** -- "Use simple language suitable for beginners"

In [ ]:
# Same question, three different system prompts — shows how persona changes output dramatically
personas = [
    ("AWS Solutions Architect", "You are an AWS Solutions Architect. Answer in bullet points. Maximum 5 points."),
    ("Exam Tutor", "You are a certification exam tutor. Explain concepts with simple analogies and concrete examples."),
    ("Code Reviewer", "You are a Python code expert. Respond only with code, no explanations. Add inline comments."),
]

question = "How do I invoke a Bedrock model using boto3?"

for name, system_prompt in personas:
    print(f"\n{'='*60}")
    print(f"Persona: {name}")
    print(f"{'='*60}")
    result = invoke(question, system=system_prompt)
    print(result)

### System Prompt Best Practices

- **Be specific** -- "Answer in bullet points" beats "Be concise"
- **Set boundaries** -- explicitly state what to include and exclude
- **Layer constraints** -- combine persona + format + tone in one system prompt
- **Test adversarial inputs** -- users may try to override the system prompt

## E. Structured Output Extraction

Production applications often need machine-parseable output (typically JSON). Reliable extraction requires:

| Technique | Why It Helps |
|-----------|-------------|
| `temperature=0` | Eliminates randomness for consistent structure |
| JSON schema in prompt | Model sees exactly what fields you expect |
| "Return ONLY valid JSON" | Prevents markdown wrapping or extra text |
| System prompt reinforcement | Doubles down on the JSON-only constraint |
| `json.loads()` validation | Catches malformed output immediately |

### Extract a Single Object

Provide the exact JSON schema in the prompt so the model knows the expected structure.

In [ ]:
# Structured extraction — provide the exact JSON schema so the model follows it
prompt = """Extract the following from this AWS service description and return ONLY valid JSON with no markdown formatting:
{
  "service_name": "",
  "category": "",
  "key_feature": "",
  "pricing_model": ""
}

Description: Amazon Bedrock is a fully managed service that offers foundation models from leading AI companies through a single API. It supports on-demand and provisioned throughput pricing."""

# temperature=0 for deterministic, consistent JSON structure
result = invoke(prompt, temperature=0.0)
print("Raw response:")
print(result)

# Strip markdown code fences if the model wraps the JSON (common with Claude)
cleaned = result.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1]
    cleaned = cleaned.rsplit("```", 1)[0]

# Validate with json.loads — in production, wrap in try/except with retry logic
parsed = json.loads(cleaned.strip())
print("\nParsed JSON:")
print(json.dumps(parsed, indent=2))

### Extract a List of Items

Extract multiple items from unstructured text into a JSON array -- a common pattern in document processing pipelines.

In [ ]:
# List extraction — pull structured data from free-text descriptions
prompt = """Extract all AWS services mentioned in the text below. For each service,
return a JSON array of objects with "service" and "purpose" keys.
Return ONLY the JSON array, no other text, no markdown formatting.

Text: Our architecture uses Amazon S3 for storing raw data files, Amazon Bedrock
for running inference on foundation models, AWS Lambda for serverless processing
of incoming requests, and Amazon CloudWatch for monitoring and logging all
API calls across the pipeline."""

result = invoke(
    prompt,
    # System prompt reinforces the JSON-only constraint at a higher priority level
    system="You are a JSON extraction engine. Return only valid JSON, no markdown, no explanation.",
    temperature=0.0
)
print("Raw response:")
print(result)

# Strip markdown code fences if present
cleaned = result.strip()
if cleaned.startswith("```"):
    cleaned = cleaned.split("\n", 1)[1]
    cleaned = cleaned.rsplit("```", 1)[0]

services = json.loads(cleaned.strip())
print(f"\nExtracted {len(services)} services:")
for svc in services:
    print(f"  - {svc['service']}: {svc['purpose']}")

### Tips for Reliable JSON Extraction

In production, always wrap `json.loads()` in a try/except and implement a retry mechanism for the rare cases where the model returns invalid JSON.

## F. Prompt Chaining

**Prompt chaining** decomposes a complex task into a pipeline of simpler steps, where each step's output feeds the next step's input.

```mermaid
flowchart LR
    D[Document] --> S1["Step 1: Summarize\n(temp=0.3)"]
    S1 --> S2["Step 2: Classify\n(temp=0)"]
    S2 --> S3["Step 3: Recommend\n(temp=0.7)"]
    S3 --> R[Results]
```

**Benefits over a single prompt:**
- **Different parameters per step** -- temperature=0 for classification, 0.7 for creative generation
- **Debuggable** -- inspect each step's output independently
- **Modular** -- swap, add, or remove steps without rewriting everything
- **Cacheable** -- skip unchanged intermediate steps in production

In [ ]:
# Prompt chaining: summarize → classify → recommend
# Each step uses a different temperature suited to its task type

document = """Amazon Bedrock is a fully managed service that makes high-performing foundation
models from leading AI companies available for your use through a unified API. You
can choose from a wide range of foundation models to find the model that is best
suited for your use case. Amazon Bedrock also offers a broad set of capabilities
to build generative AI applications with security, privacy, and responsible AI.
Using Amazon Bedrock, you can easily experiment with and evaluate top foundation
models for your use cases, privately customize them with your data using techniques
such as fine-tuning and Retrieval Augmented Generation (RAG), and build agents that
execute tasks using your enterprise systems and data sources. Since Amazon Bedrock
is serverless, you don't have to manage any infrastructure, and you can securely
integrate and deploy generative AI capabilities into your applications using the
AWS services you are already familiar with."""

# Step 1: Summarize — low temperature for faithful condensation
print("STEP 1 — Summarize")
print("-" * 40)
summary = invoke(f"Summarize this in 2 sentences:\n\n{document}", temperature=0.3)
print(f"{summary}\n")

# Step 2: Classify — temperature=0 for deterministic single-label output
print("STEP 2 — Classify")
print("-" * 40)
classification = invoke(
    f"Classify this summary into exactly one category (compute/storage/ai-ml/database/networking). "
    f"Respond with only the category name.\n\n{summary}",
    temperature=0.0
)
print(f"Category: {classification.strip()}\n")

# Step 3: Recommend — higher temperature for creative suggestions
print("STEP 3 — Recommend")
print("-" * 40)
recommendation = invoke(
    f"Given this {classification.strip()} service summary, suggest 3 complementary AWS services "
    f"that would work well together in a production architecture. For each, explain why in one sentence.\n\n{summary}",
    temperature=0.7
)
print(f"Recommendations:\n{recommendation}")

### Chaining vs Single Prompt

You could try "summarize, classify, and recommend" in one prompt, but chaining wins because:

- **Step 2 uses temperature=0** (classification) while **Step 3 uses 0.7** (creative) -- impossible in a single call
- Wrong classification? **Fix Step 2 independently** without regenerating the summary
- Each step's output is **inspectable** -- log and debug each stage
- In production, **cache intermediate results** -- skip summarization if the document hasn't changed

## Key Takeaways

1. Prompt engineering follows a natural progression: **zero-shot** (simplest) → **few-shot** (add examples) → **chain-of-thought** (add reasoning) → **chaining** (decompose into steps)
2. Set **temperature=0** for deterministic tasks like classification, extraction, and factual Q&A; use higher values for creative generation
3. **System prompts** control model persona and output constraints — they are the primary tool for customizing model behavior in production applications
4. **Structured output** requires explicit format instructions, a JSON schema in the prompt, and temperature=0 for reliability
5. **Prompt chaining** decomposes complex tasks into focused steps, enabling different parameters per step and easier debugging

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Zero-Shot Prompting | Giving the model a task with no examples; relies entirely on pre-training knowledge |
| Few-Shot Prompting | Providing 2-3 input-output examples in the prompt to teach the model the expected format and behavior |
| Chain-of-Thought (CoT) | Instructing the model to show step-by-step reasoning before giving a final answer, improving accuracy on multi-step tasks |
| System Prompt | A dedicated instruction set (via the Converse API `system` parameter) that controls model persona, constraints, and behavior |
| Structured Output | Prompting the model to return data in a machine-parseable format like JSON, using explicit schemas and temperature=0 |
| Prompt Template | A reusable prompt with placeholder variables (e.g., `{description}`) that gets filled in at runtime for consistent formatting |
| Prompt Chaining | Breaking a complex task into a pipeline of sequential model calls, where each step's output feeds into the next step's input |

## Exam Preparation — Common Exam Question Patterns

**Q: Which prompting technique improves accuracy on multi-step reasoning tasks?**
A: Chain-of-thought (CoT) prompting. By forcing the model to show step-by-step reasoning, each intermediate step provides context for the next, reducing errors on math, logic, and planning tasks.

**Q: How do you ensure consistent JSON output from a foundation model?**
A: Use temperature=0 for deterministic output, include the exact JSON schema in the prompt, add an explicit instruction like "return ONLY valid JSON", and optionally reinforce with a system prompt. Always validate the output with `json.loads()`.

**Q: What is the difference between zero-shot and few-shot prompting?**
A: Zero-shot provides no examples — the model relies on pre-training. Few-shot includes 2-3 examples in the prompt that demonstrate the expected input-output format. Few-shot produces more consistent and predictable outputs, especially for classification and extraction tasks.

**Q: How does the Converse API system parameter differ from putting instructions in the user message?**
A: The `system` parameter sets persistent behavioral instructions that are processed separately from user input. System prompts have stronger influence over model behavior and are ideal for persona definition, output constraints, and guardrails. User-message instructions may be overridden by conversational context.

**Q: When should you use prompt chaining instead of a single prompt?**
A: Use prompt chaining when the task involves multiple distinct steps that benefit from different parameters (e.g., temperature=0 for classification, temperature=0.7 for generation), when you need to inspect or debug intermediate results, or when the combined task is too complex for a single prompt to handle reliably.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Bedrock API — Zero-shot (Claude Sonnet 4.5) | ~2K input + 2K output tokens | ~$0.05 |
| Bedrock API — Few-shot (Claude Sonnet 4.5) | ~3K input + 1K output tokens | ~$0.04 |
| Bedrock API — Chain-of-thought (Claude Sonnet 4.5) | ~2K input + 3K output tokens | ~$0.06 |
| Bedrock API — System prompts (Claude Sonnet 4.5) | ~3K input + 3K output tokens | ~$0.06 |
| Bedrock API — Structured output (Claude Sonnet 4.5) | ~2K input + 1K output tokens | ~$0.03 |
| Bedrock API — Prompt chaining (Claude Sonnet 4.5) | ~4K input + 3K output tokens | ~$0.07 |
| **Total** | | **~$0.31** |

All API calls use Claude Sonnet 4.5 via the Converse API. Total cost well under $1.